In [4]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd

files = [
    Path(f) for f in glob.glob("sim_stats/*/*/*/ue_stats.csv")
    if 1 <= int(Path(f).parent.name.removeprefix("SEED")) <= 5
]
dfs = []

for file in files:
    df = pd.read_csv(file)
    df["policy"] = file.parents[2].name
    df["scenario"] = file.parents[1].name
    dfs.append(df)

if not dfs:
    raise ValueError("No ue_stats.csv files matched the selected policies and seeds")
df = pd.concat(dfs, ignore_index=True)

required = {"completed_packets", "dropped_packets", "delay_budget"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"ue_stats.csv missing columns: {sorted(missing)}")

packet_counts = df[["completed_packets", "dropped_packets"]]
if not np.isfinite(packet_counts.to_numpy(dtype=float)).all():
    raise ValueError("packet counts must contain only finite values")
if (packet_counts < 0).any().any():
    raise ValueError("packet counts cannot be negative")

df["total_packets"] = (
    df["completed_packets"] + df["dropped_packets"]
).astype(int)

# A UE leaves the simulator only after its packets have outcomes.  The number
# of packets per UE is configurable, so aggregate the recorded packet counts.
if (df["total_packets"] <= 0).any():
    raise ValueError("Each completed UE must contain at least one packet outcome")

stats = (
    df.groupby(["policy", "scenario", "delay_budget"])
    .agg(
        total_ues=("total_packets", "size"),
        total_packets=("total_packets", "sum"),
        completed_packets=("completed_packets", "sum"),
        dropped_packets=("dropped_packets", "sum"),
    )
)

# Percentage, not a fraction.
stats["drop_ratio"] = (
    100 * stats["dropped_packets"] / stats["total_packets"]
)

print(stats.to_string(float_format=lambda x: f"{x:.4f}"))


                                    total_ues  total_packets  completed_packets  dropped_packets  drop_ratio
policy       scenario delay_budget                                                                          
A1000        A        0.05                 24           1200               1195                5      0.4167
                      0.15               2535         126750             126714               36      0.0284
                      0.30               4116         205800             205727               73      0.0355
B220_1000    B        0.05                161           8050               8028               22      0.2733
                      0.15              13827         691350             690042             1308      0.1892
                      0.30              22185        1109250            1108203             1047      0.0944
B220_1500    B        0.05                383          19150              19092               58      0.3029
                   

In [97]:
from pathlib import Path
import numpy as np
import pandas as pd

policy = "C*"


files = sorted(
    Path("sim_stats").glob(f"{policy}/*/*/bs_stats.csv"),
    key=lambda p: int(p.parent.name.removeprefix("SEED"))
)

rows = []
all_times = np.zeros(4)

for file in files:
    df = pd.read_csv(file)

    # sleep_time 顺序：Active、SM1、SM2、SM3
    times = np.stack([
        np.fromstring(str(value).strip("[]"), sep=" ")
        for value in df["sleep_time"]
    ])

    if times.shape[1] != 4 or not np.isfinite(times).all():
        raise ValueError(f"sleep_time 格式错误：{file}")
    if (times < 0).any() or times.sum() <= 0:
        raise ValueError(f"sleep_time 数值错误：{file}")
    # 汇总所有基站的驻留时间，再计算比例
    seed_times = times.sum(axis=0)
    all_times += seed_times

    rows.append([
        file.parent.name,
        *(seed_times / seed_times.sum() * 100)
    ])

rows.append(["整体", *(all_times / all_times.sum() * 100)])

result = pd.DataFrame(
    rows,
    columns=["seed", "Active (%)", "SM1 (%)", "SM2 (%)", "SM3 (%)"]
).set_index("seed")

print(result.to_string(float_format=lambda x: f"{x:.3f}"))

       Active (%)  SM1 (%)  SM2 (%)  SM3 (%)
seed                                        
SEED1      29.537    3.648   66.608    0.207
SEED1      55.519    0.207   44.275    0.000
SEED2      28.914    2.005   68.964    0.118
SEED2      55.175    0.179   44.646    0.000
SEED3      28.743    5.171   65.990    0.095
SEED3      55.403    0.158   44.439    0.000
SEED4      55.422    0.191   44.387    0.000
SEED5      55.218    0.198   44.584    0.000
整体         45.491    1.470   52.987    0.053
